In [78]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [79]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


target = converted column

In [80]:
data.shape

(5000, 9)

In [81]:
# features = data.drop(columns='converted')
# features.head()

In [82]:
# target = data['converted']
# target.head()

In [83]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [84]:
cat_cols = list(df.select_dtypes(include=["object"]).columns)
cat_cols

/var/folders/yy/t95v8nfd5mg826zr_b6pmwf00000gp/T/ipykernel_39858/2188456431.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = list(df.select_dtypes(include=["object"]).columns)


['lead_source', 'industry', 'employment_status', 'location']

In [85]:
num_cols = list(df.select_dtypes(include=["number"]).columns)
num_cols

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [86]:
## replace cat cols num values with NA
df[cat_cols] = df[cat_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

In [87]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [88]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [89]:
df[num_cols].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [90]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


In [91]:
from sklearn.metrics import mutual_info_score

In [92]:
def mutual_info_converted_score(series):
    return round(mutual_info_score(series, df_train.converted) , 2)


In [93]:
mi = df_train[cat_cols].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [94]:
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']
df_train = df_train.drop(columns="converted")
df_val = df_val.drop(columns="converted")
df_test = df_test.drop(columns="converted")

In [95]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [96]:
preprocessor = ColumnTransformer(
    transformers =  [
        ("cat", OneHotEncoder(drop="first",sparse_output=False, handle_unknown="ignore"), cat_cols)
    ], remainder="passthrough"
)
preprocessor.set_output(transform="pandas")

df_train = preprocessor.fit_transform(df_train) 
df_val = preprocessor.transform(df_val) 
df_test = preprocessor.transform(df_test) 

In [97]:
df_train.head()

,cat__lead_source_events,cat__lead_source_organic_search,cat__lead_source_paid_ads,cat__lead_source_referral,cat__lead_source_social_media,cat__industry_education,cat__industry_finance,cat__industry_healthcare,cat__industry_manufacturing,cat__industry_retail,...,cat__employment_status_unemployed,cat__location_africa,cat__location_asia,cat__location_europe,cat__location_north_america,cat__location_south_america,remainder__annual_income,remainder__number_of_courses_viewed,remainder__interaction_count,remainder__lead_score
4702,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,65980.0,2,3,0.37
2155,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,71662.0,0,0,0.09
289,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,40220.0,2,5,0.55
1822,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,88352.0,2,2,0.45
3442,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,89316.0,1,5,0.00


In [98]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(df_train, y_train)
y_pred = model.predict(df_val)

In [100]:
from sklearn.metrics import accuracy_score

score = accuracy_score(y_pred, y_val)
round(score,2)

0.65

In [101]:
orig_model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

orig_model.fit(df_train, y_train)
y_pred = orig_model.predict(df_val)

orig_score = accuracy_score(y_pred, y_val)

In [ ]:
df_train.columns

Index(['cat__lead_source_events', 'cat__lead_source_organic_search',
       'cat__lead_source_paid_ads', 'cat__lead_source_referral',
       'cat__lead_source_social_media', 'cat__industry_education',
       'cat__industry_finance', 'cat__industry_healthcare',
       'cat__industry_manufacturing', 'cat__industry_retail',
       'cat__industry_technology', 'cat__employment_status_employed',
       'cat__employment_status_self_employed',
       'cat__employment_status_student', 'cat__employment_status_unemployed',
       'cat__location_africa', 'cat__location_asia', 'cat__location_europe',
       'cat__location_north_america', 'cat__location_south_america',
       'remainder__annual_income', 'remainder__number_of_courses_viewed',
       'remainder__interaction_count', 'remainder__lead_score'],
      dtype='str')

In [108]:
scores = []

for col in df_train.columns:
    # print(col)
    features = [feat for feat in df_train.columns if  feat!=col]
    df_local = df_train[features]
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(df_local, y_train)
    y_pred = model.predict(df_val[features])
    score = accuracy_score(y_val, y_pred)
    scores.append([col, orig_score - score])
print(scores)

[['cat__lead_source_events', -0.0020000000000000018], ['cat__lead_source_organic_search', -0.0010000000000000009], ['cat__lead_source_paid_ads', 0.0020000000000000018], ['cat__lead_source_referral', 0.0], ['cat__lead_source_social_media', -0.0010000000000000009], ['cat__industry_education', -0.0010000000000000009], ['cat__industry_finance', 0.0], ['cat__industry_healthcare', -0.0010000000000000009], ['cat__industry_manufacturing', 0.0010000000000000009], ['cat__industry_retail', -0.0010000000000000009], ['cat__industry_technology', -0.0010000000000000009], ['cat__employment_status_employed', 0.0], ['cat__employment_status_self_employed', 0.0], ['cat__employment_status_student', -0.0010000000000000009], ['cat__employment_status_unemployed', -0.0010000000000000009], ['cat__location_africa', 0.0010000000000000009], ['cat__location_asia', -0.0010000000000000009], ['cat__location_europe', 0.0], ['cat__location_north_america', -0.0010000000000000009], ['cat__location_south_america', 0.0], ['

In [109]:
scores_df = pd.DataFrame(scores,columns=["Feature removed", "score change"])

scores_df.sort_values("score change", ascending=False)
scores_df

,Feature removed,score change
0,cat__lead_source_events,-0.002
1,cat__lead_source_organic_search,-0.001
2,cat__lead_source_paid_ads,0.002
3,cat__lead_source_referral,0.000
4,cat__lead_source_social_media,-0.001
5,cat__industry_education,-0.001
6,cat__industry_finance,0.000
7,cat__industry_healthcare,-0.001
8,cat__industry_manufacturing,0.001
9,cat__industry_retail,-0.001


In [110]:
from sklearn.linear_model import LogisticRegression



for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)

    model.fit(df_train, y_train)
    y_pred = model.predict(df_val)
    score = round(accuracy_score(y_val, y_pred), 3)
    print(f"C: {c}, Accuracy: {score}")


C: 1e-06, Accuracy: 0.598
C: 1e-05, Accuracy: 0.598
C: 0.0001, Accuracy: 0.613
C: 0.001, Accuracy: 0.645
